IMPORTING DATA + INSTALLING REQUIRED PACKAGES

In [8]:
from pathlib import Path
import pandas as pd

def find_project_root(marker=".git", start=None):
    """Walk up from the current directory until we find the repo root
    (identified by the presence of a .git folder)."""
    current = Path(start or Path.cwd()).resolve()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Could not locate project root (no {marker} found above {current})."
    )

PROJECT_DIR = find_project_root()
DATA_DIR = PROJECT_DIR / "data" / "raw"
SUBMISSION_DIR = PROJECT_DIR / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

def resolve_data_file(canonical_name, fallback_name):
    """Prefer data/raw; retain compatibility with the current root-level download."""
    canonical_path = DATA_DIR / canonical_name
    fallback_path = DATA_DIR / fallback_name  # fixed: was PROJECT_DIR
    if canonical_path.exists():
        return canonical_path
    if fallback_path.exists():
        print(f"Using fallback file: {fallback_path.name}")
        return fallback_path
    raise FileNotFoundError(
        f"Could not find {canonical_path} or {fallback_path}. "
        "Place the competition CSVs in data/raw."
    )

TRAIN_PATH = resolve_data_file("train.csv", "train (1).csv")
TEST_PATH = resolve_data_file("test.csv", "test (1).csv")
SAMPLE_SUBMISSION_PATH = resolve_data_file(
    "sample_submission.csv", "sample_submission.csv"
)

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

test_passenger_ids = test_df["PassengerId"].copy()
train_df["is_train"] = True
test_df["is_train"] = False
combined_raw = pd.concat([train_df, test_df], ignore_index=True, sort=False)

print(f"Project root: {PROJECT_DIR}")
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Combined shape: {combined_raw.shape}")
display(sample_submission.head())

Using fallback file: train (1).csv
Using fallback file: test (1).csv
Project root: /workspaces/alpha-centauri-rescue
Train shape: (8693, 15)
Test shape: (4277, 14)
Combined shape: (12970, 15)


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [9]:
import numpy as np, pandas as pd
from itertools import combinations
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold
from sklearn.metrics import accuracy_score

SEED = 42                      # shared by the whole team
USE_STRATIFIED = False         # True -> StratifiedGroupKFold with the SAME SEED for everyone
USE_SURNAME_FILL = True        # uses other passengers' features only; set False to be strict
TARGET = "Transported"
SPEND = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

# =====================================================================
# STAGE A: stateless, label-free (target is dropped immediately)
# =====================================================================
def build_static_features(train_df, test_df=None):
    full = pd.concat([train_df, test_df] if test_df is not None else [train_df],
                     ignore_index=True)
    n_train = len(train_df)
    y = full.loc[:n_train - 1, TARGET].map({True: 1, False: 0, "True": 1, "False": 0}).astype(int)
    full = full.drop(columns=TARGET)          # <- from here on the target does not exist

    for c in ["CryoSleep", "VIP"]:
        full[c] = full[c].map({True: 1.0, False: 0.0, "True": 1.0, "False": 0.0})

    # parse
    full["Group"] = full["PassengerId"].str.split("_").str[0].astype(int)
    full["GroupSize"] = full.groupby("Group")["Group"].transform("count")
    full["Solo"] = (full["GroupSize"] == 1).astype(int)
    cab = full["Cabin"].str.split("/", expand=True)
    full["Deck"], full["CabinNum"], full["Side"] = cab[0], pd.to_numeric(cab[1], errors="coerce"), cab[2]
    full["CabinOccupancy"] = full.groupby("Cabin")["Cabin"].transform("count").fillna(1)
    full["Surname"] = full["Name"].str.split().str[-1]
    full["NumMissing"] = full[["HomePlanet", "CryoSleep", "Cabin", "Destination", "Age",
                               "VIP", "Name"] + SPEND].isna().sum(axis=1)

    # RULE 1: CryoSleep first, then spend = 0 for cryosleepers (and children)
    spend_known = full[SPEND].notna().all(axis=1)
    spend_sum = full[SPEND].sum(axis=1)
    full.loc[full["CryoSleep"].isna() & (spend_sum > 0), "CryoSleep"] = 0.0
    full.loc[full["CryoSleep"].isna() & spend_known & (spend_sum == 0), "CryoSleep"] = 1.0
    zero_spend = (full["CryoSleep"] == 1) | (full["Age"] < 13)
    for c in SPEND:
        full.loc[full[c].isna() & zero_spend, c] = 0.0
    # remaining spend NaNs are imputed in Stage B (needs a fitted median)
    for c in SPEND:
        full.loc[full[c].isna() & zero_spend, c] = 0.0

    # NEW: label-free group aggregates
    spend_total = full[SPEND].sum(axis=1)
    full["GroupSpendMean"] = spend_total.groupby(full["Group"]).transform("mean")
    full["GroupCryoShare"] = full["CryoSleep"].groupby(full["Group"]).transform("mean").fillna(-1)
    full["GroupAgeMean"] = full["Age"].groupby(full["Group"]).transform("mean").fillna(-1)

    # group-mate / surname imputation ...

    # group-mate / surname imputation (other passengers' FEATURES only, never the target)
    def fill_by(col, key):
        f = full.groupby(key)[col].transform(lambda s: s.dropna().mode().iloc[0]
                                              if s.notna().any() else np.nan)
        full[col] = full[col].fillna(f)
    for col in ["HomePlanet", "Deck", "Side", "CabinNum", "Destination"]:
        fill_by(col, "Group")
    if USE_SURNAME_FILL:
        fill_by("HomePlanet", "Surname")
    full.loc[full["HomePlanet"].isna() & full["Deck"].isin(list("ABCT")), "HomePlanet"] = "Europa"
    full.loc[full["HomePlanet"].isna() & (full["Deck"] == "G"), "HomePlanet"] = "Earth"

    key = full["Surname"].fillna("?") + "_" + full["HomePlanet"].fillna("?")
    full["FamilySize"] = key.groupby(key).transform("count")
    full.loc[full["Surname"].isna(), "FamilySize"] = 1

    X = full.iloc[:n_train].reset_index(drop=True)
    X_test = full.iloc[n_train:].reset_index(drop=True) if test_df is not None else None
    assert TARGET not in X.columns
    return X, X_test, y.reset_index(drop=True), X["Group"]

# =====================================================================
# STAGE B: fitted on TRAINING rows of each fold only
# =====================================================================
CAT_FILL = ["HomePlanet", "Destination", "Deck", "Side", "VIP", "CryoSleep"]
NUM_COLS = (["Age", "CabinNum", "GroupSize", "Solo", "CabinOccupancy", "FamilySize",
             "NumMissing", "CryoSleep", "VIP", "NoSpend", "SpendCategories", "IsChild",
             "GroupSpendMean", "GroupCryoShare", "GroupAgeMean"]
            + [f"{c}_log" for c in SPEND + ["TotalSpend"]])
CAT_COLS = ["HomePlanet", "Destination", "Deck", "Side", "AgeBin", "CabinRegion"]

class SpendAgePrep(BaseEstimator, TransformerMixin):
    """Learns modes/medians/clip bounds in fit(); applies them in transform()."""
    def __init__(self, q=(0.01, 0.99)):
        self.q = q

    def _impute(self, X):
        X = X.copy()
        for c, v in self.modes_.items():
            X[c] = X[c].fillna(v)
        X["CabinNum"] = X["CabinNum"].fillna(X["Deck"].map(self.cabin_med_)).fillna(self.cabin_med_all_)
        idx = pd.MultiIndex.from_frame(X[["HomePlanet", "Deck"]])
        age_fill = pd.Series(self.age_med_.reindex(idx).values, index=X.index)
        X["Age"] = X["Age"].fillna(age_fill).fillna(self.age_med_all_)
        zero_spend = (X["CryoSleep"] == 1) | (X["Age"] < 13)
        for c in SPEND:
            X.loc[X[c].isna() & zero_spend, c] = 0.0       # CryoSleep rule first
            X[c] = X[c].fillna(self.spend_med_[c])          # then impute the rest
        return X

    def fit(self, X, y=None):                               # y is deliberately unused
        self.modes_ = {c: X[c].mode()[0] for c in CAT_FILL}
        self.cabin_med_ = X.groupby("Deck")["CabinNum"].median()
        self.cabin_med_all_ = X["CabinNum"].median()
        tmp = X.copy()
        for c in ["HomePlanet", "Deck"]:
            tmp[c] = tmp[c].fillna(self.modes_[c])
        self.age_med_ = tmp.groupby(["HomePlanet", "Deck"])["Age"].median()
        self.age_med_all_ = X["Age"].median()
        self.spend_med_ = {c: X.loc[X["CryoSleep"] == 0, c].median() for c in SPEND}
        imp = self._impute(X)
        self.bounds_ = {c: tuple(imp[c].quantile(self.q)) for c in SPEND}   # fold-wise p1/p99
        return self

    def transform(self, X):
        X = self._impute(X)
        for c in SPEND:
            X[c] = X[c].clip(*self.bounds_[c])              # clip, THEN log1p
        X["TotalSpend"] = X[SPEND].sum(axis=1)
        X["NoSpend"] = (X["TotalSpend"] == 0).astype(int)
        X["SpendCategories"] = (X[SPEND] > 0).sum(axis=1)
        for c in SPEND + ["TotalSpend"]:
            X[f"{c}_log"] = np.log1p(X[c])
        X["IsChild"] = (X["Age"] < 13).astype(int)
        X["AgeBin"] = pd.cut(X["Age"], [-0.1, 12, 17, 25, 32, 50, np.inf],
                             labels=["child", "teen", "young_adult", "adult",
                                     "middle_age", "senior"]).astype(str)
        X["CabinRegion"] = "R" + (X["CabinNum"] // 300).astype(int).astype(str)
        return X[NUM_COLS + CAT_COLS]

def make_pipeline(model):
    enc = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS)],
        remainder="passthrough")
    return Pipeline([("prep", SpendAgePrep()), ("enc", enc), ("model", model)])

# =====================================================================
# Shared folds, models, OOF predictions, blends
# =====================================================================
X, X_test, y, groups = build_static_features(train_df, test_df)   # test_df: your test.csv frame

splitter = (StratifiedGroupKFold(5, shuffle=True, random_state=SEED) if USE_STRATIFIED
            else GroupKFold(5))
splits = list(splitter.split(X, y, groups))
fold_id = np.zeros(len(X), int)
for k, (_, va) in enumerate(splits):
    fold_id[va] = k
pd.DataFrame({"PassengerId": X["PassengerId"], "fold": fold_id}).to_csv("folds.csv", index=False)
# -> share folds.csv so everyone evaluates on identical splits

def get_models():
    m = {"hgb": HistGradientBoostingClassifier(random_state=SEED)}
    try:
        from xgboost import XGBClassifier
        m["xgb"] = XGBClassifier(n_estimators=400, learning_rate=0.03, max_depth=4,
                                 subsample=0.8, colsample_bytree=0.8, random_state=SEED,
                                 eval_metric="logloss")
    except ImportError: pass
    try:
        from lightgbm import LGBMClassifier
        m["lgbm"] = LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15,
                                   subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                   random_state=SEED, verbose=-1)
    except ImportError: pass
    try:
        from catboost import CatBoostClassifier
        m["cat"] = CatBoostClassifier(iterations=600, learning_rate=0.03, depth=5,
                                      random_seed=SEED, verbose=0)
    except ImportError: pass
    return m

def fold_acc(p):
    return np.array([accuracy_score(y.iloc[va], p[va] > 0.5) for _, va in splits])

oof = {}
for name, model in get_models().items():
    p = np.zeros(len(X))
    for tr, va in splits:
        pipe = make_pipeline(clone(model)).fit(X.iloc[tr], y.iloc[tr])
        p[va] = pipe.predict_proba(X.iloc[va])[:, 1]
    oof[name] = p
    s = fold_acc(p)
    print(f"{name:5s} CV {s.mean():.4f} +/- {s.std():.4f}")

rows = []
for k in range(2, len(oof) + 1):
    for combo in combinations(oof, k):
        s = fold_acc(np.mean([oof[n] for n in combo], axis=0))
        rows.append(("+".join(combo), s.mean(), s.std()))
print(pd.DataFrame(rows, columns=["blend", "cv_mean", "cv_std"])
        .sort_values("cv_mean", ascending=False).round(4).to_string(index=False))

# =====================================================================
# Final fit on all training rows -> submission (LB is a sanity check only)
# =====================================================================
BEST = ["cat", "xgb", "hgb"]                      # set to the best CV blend printed above
models = get_models()
probs = [make_pipeline(clone(models[n])).fit(X, y).predict_proba(X_test)[:, 1] for n in BEST]
sub = pd.DataFrame({"PassengerId": X_test["PassengerId"],
                    TARGET: np.mean(probs, axis=0) > 0.5})
sub.to_csv("submission.csv", index=False)

hgb   CV 0.8087 +/- 0.0117
xgb   CV 0.8078 +/- 0.0075
lgbm  CV 0.8066 +/- 0.0131
cat   CV 0.8096 +/- 0.0057
           blend  cv_mean  cv_std
    xgb+lgbm+cat   0.8098  0.0088
         hgb+cat   0.8097  0.0078
         xgb+cat   0.8096  0.0085
        hgb+lgbm   0.8096  0.0105
     hgb+xgb+cat   0.8094  0.0075
hgb+xgb+lgbm+cat   0.8093  0.0078
        xgb+lgbm   0.8092  0.0084
    hgb+lgbm+cat   0.8092  0.0088
    hgb+xgb+lgbm   0.8087  0.0092
         hgb+xgb   0.8085  0.0080
        lgbm+cat   0.8080  0.0089


In [10]:
from sklearn.metrics import log_loss

def fold_ll(p):
    return np.array([log_loss(y.iloc[va], p[va]) for _, va in splits])

rows = []
for k in range(1, len(oof) + 1):
    for combo in combinations(oof, k):
        p = np.mean([oof[n] for n in combo], axis=0)
        a, l = fold_acc(p), fold_ll(p)
        rows.append(("+".join(combo), a.mean(), a.std(), l.mean(), l.std()))
rank = pd.DataFrame(rows, columns=["model", "acc", "acc_std", "logloss", "ll_std"])
print(rank.sort_values("logloss").round(4).to_string(index=False))

           model    acc  acc_std  logloss  ll_std
    hgb+lgbm+cat 0.8092   0.0088   0.3809  0.0135
         hgb+cat 0.8097   0.0078   0.3812  0.0129
        hgb+lgbm 0.8096   0.0105   0.3814  0.0164
hgb+xgb+lgbm+cat 0.8093   0.0078   0.3825  0.0133
    hgb+xgb+lgbm 0.8087   0.0092   0.3830  0.0151
        lgbm+cat 0.8080   0.0089   0.3831  0.0117
     hgb+xgb+cat 0.8094   0.0075   0.3831  0.0128
            lgbm 0.8066   0.0131   0.3832  0.0150
         hgb+xgb 0.8085   0.0080   0.3843  0.0153
    xgb+lgbm+cat 0.8098   0.0088   0.3848  0.0120
             hgb 0.8087   0.0117   0.3856  0.0184
        xgb+lgbm 0.8092   0.0084   0.3858  0.0140
             cat 0.8096   0.0057   0.3866  0.0088
         xgb+cat 0.8096   0.0085   0.3870  0.0107
             xgb 0.8078   0.0075   0.3906  0.0131


In [11]:
from sklearn.compose import ColumnTransformer

class ColDrop(BaseEstimator, TransformerMixin):
    def __init__(self, cols=()):
        self.cols = cols
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X.drop(columns=list(self.cols))

def make_pipeline_v2(model, drop=()):
    enc = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
          lambda X: [c for c in CAT_COLS if c in X.columns])],
        remainder="passthrough")
    return Pipeline([("prep", SpendAgePrep()), ("drop", ColDrop(drop)),
                     ("enc", enc), ("model", model)])

def cv_eval(model_name, drop=()):
    m = get_models()[model_name]
    p = np.zeros(len(X))
    for tr, va in splits:
        pipe = make_pipeline_v2(clone(m), drop).fit(X.iloc[tr], y.iloc[tr])
        p[va] = pipe.predict_proba(X.iloc[va])[:, 1]
    return p

ABLATE = {
    "family":   ["FamilySize"],
    "group":    ["GroupSize", "Solo"],
    "cabin":    ["CabinNum", "CabinOccupancy", "CabinRegion"],
    "flags":    ["NoSpend", "SpendCategories", "IsChild", "NumMissing"],
    "age_bin":  ["AgeBin"],
    "item_logs": [f"{c}_log" for c in SPEND],     # keep only TotalSpend_log
}

base = cv_eval("hgb")
b_acc, b_ll = fold_acc(base), fold_ll(base)
out = [("baseline", b_acc.mean(), b_ll.mean(), 0.0, 0)]
for name, cols in ABLATE.items():
    p = cv_eval("hgb", cols)
    a, l = fold_acc(p), fold_ll(p)
    out.append((f"drop {name}", a.mean(), l.mean(),
                (a - b_acc).mean(), int((l < b_ll).sum())))
print(pd.DataFrame(out, columns=["variant", "acc", "logloss",
                                 "d_acc_vs_base", "folds_where_ll_better"])
        .round(4).to_string(index=False))

       variant    acc  logloss  d_acc_vs_base  folds_where_ll_better
      baseline 0.8087   0.3856         0.0000                      0
   drop family 0.8085   0.3856        -0.0002                      2
    drop group 0.8059   0.3875        -0.0028                      1
    drop cabin 0.8031   0.4092        -0.0056                      0
    drop flags 0.8117   0.3866         0.0030                      2
  drop age_bin 0.8073   0.3870        -0.0014                      1
drop item_logs 0.7538   0.4890        -0.0549                      0
